# שיעור 10 - סוכני AI בסביבת ייצור

בשיעור זה תלמדו **תבניות ליישום בייצור** עבור סוכני AI באמצעות Microsoft Agent Framework (Python). אנו מכסים:

- **תצפית (Observability)** — הוספת מדידת זמן ורישום לאינטראקציות של הסוכן
- **הערכה (Evaluation)** — שימוש בסוכן מעריך כדי לדרג את איכות התגובות
- **ניהול עלויות (Cost management)** — אסטרטגיות לאופטימיזציה של טוקנים ולבחירת מודלים

התסריט הוא **סוכן נסיעות** שעוזר למשתמשים לתכנן טיולים, עם ניטור והערכה שמתווספים מעליו.


## הגדרה


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## שיקולים בסביבת ייצור

העברת סוכני בינה מלאכותית מפרוטוטיפים לסביבת הייצור דורשת תשומת לב קפדנית לשלושה עקרונות מרכזיים:

1. **ניטור ותצפית** — יש צורך ביכולת לראות מה הסוכן עושה, כמה זמן זה לוקח ואילו כלים הוא מפעיל. ללא כלי מעקב ורישום, איתור תקלות בסביבת הייצור כמעט בלתי אפשרי.

2. **הערכה** — בדיקות איכות אוטומטיות מבטיחות שהתשובות של הסוכן נשארות מדויקות, שלמות ומועילות לאורך זמן. סוכן מעריך יכול לדרג תשובות בהתאם לקריטריונים מוגדרים.

3. **ניהול עלויות** — צריכת טוקנים משפיעה ישירות על העלות. אסטרטגיות כמו אופטימיזציה של הפרומפטים, בחירת מודל ומטמון עוזרות לשמור על הוצאות תחת שליטה מבלי לפגוע באיכות.


## בניית סוכן הניתן לתצפית

אנו מגדירים כלים לנסיעות ועוטפים את קריאת הסוכן במדידת זמן כדי שנוכל לנטר את השהייה. בסביבת הייצור תשתלבו עם OpenTelemetry או עם מערכת מעקב דומה.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## דפוסי הערכה

דפוס שכיח במערכות ייצור הוא להשתמש בסוכן שני כ-**מעריך**.

המעריך מדרג את תגובת הסוכן הראשי בהתאם לקריטריונים שהוגדרו מראש כגון שלמות, דיוק ומועילות.

זה מאפשר:

- שערי איכות אוטומטיים לפני שהתגובות מגיעות למשתמשים
- זיהוי רגרסיה כאשר ההנחיות או המודלים משתנים
- ניטור מתמשך של ביצועי הסוכן לאורך זמן


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## אסטרטגיות לניהול עלויות

שליטה בעלויות קריטית עבור סוכני AI בסביבת ייצור. להלן אסטרטגיות מרכזיות:

| אסטרטגיה | תיאור |
|---|---|
| **אופטימיזציה של הפרומפט** | שמרו על הוראות המערכת תמציתיות. הסירו הקשרים מיותרים כדי לצמצם את מספר הטוקנים בקלט. |
| **בחירת מודל** | השתמשו במודלים קטנים וזולים יותר (למשל GPT-4o-mini) למשימות פשוטות כמו סיווג או חילוץ, והשאירו מודלים גדולים יותר למשימות של הסקת מסקנות מורכבות. |
| **מטמון** | שמרו בתור מטמון תוצאות כלים ושאילתות תכופות כדי להימנע מקריאות API מיותרות. |
| **תקציבי טוקנים** | הגדירו גבולות של `max_tokens` כדי למנוע תשובות ארוכות בלתי צפויות. |
| **אצווה** | קבצו מספר שאילתות של משתמשים לקריאת API יחידה כאשר אפשרי. |

בפועל, גישה רב-שכבתית עובדת היטב: ניתבו בקשות פשוטות למודל מהיר וזול והעבירו רק שאילתות מורכבות למודל בעל יכולת גבוהה יותר.


## סיכום

בשיעור זה למדת כיצד:

1. **הוסף תצפיתיות** לאינטראקציות של הסוכנים באמצעות מדידת זמנים ורישום, ובכך יסדת את הבסיס למעקב ולניטור.
2. **הערך תגובות של הסוכן** באופן אוטומטי באמצעות סוכן מעריך המדרג שלמות, דיוק ושימושיות.
3. **נהל עלויות** באמצעות אופטימיזציה של ההנחיות, בחירת מודלים, שמירה במטמון ותקציבי אסימונים.

דפוסי ייצור אלה מסייעים להבטיח שסוכני הבינה המלאכותית שלך יהיו אמינים, ניתנים למדידה וחסכוניים מבחינה עלותית בקנה מידה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
הצהרת אי-אחריות:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית Co-op Translator (https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש להביא בחשבון כי תרגומים אוטומטיים עלולים להכיל שגיאות או אי־דיוקים. יש לראות במסמך המקורי בשפת המקור כמקור הסמכות. עבור מידע קריטי מומלץ תרגום מקצועי על ידי מתרגם אנושי. איננו נושאים באחריות לכל אי־הבנה או פרשנות שגויה הנובעת מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
